# Project 01: Fashion-MNIST Image Classification (NeuroGebra Native)

This project trains an image classifier with NeuroGebra `ModelBuilder` and uses Training Observatory logs.


## Dataset Source and Download Instructions

- Dataset: Fashion-MNIST (Zalando Research)
- Official source: https://github.com/zalandoresearch/fashion-mnist
- License: MIT
- Auto-download in this notebook: `torchvision.datasets.FashionMNIST(download=True)`
- Manual fallback:
  1. Download from the official source above.
  2. Place extracted files under `./data/FashionMNIST/raw/`.
  3. Re-run the data loading cell.


In [ ]:
%pip -q install neurogebra torch torchvision matplotlib scikit-learn


In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report
from torchvision import datasets, transforms

from neurogebra.builders.model_builder import ModelBuilder

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
data_root = Path("./data")

train_ds = datasets.FashionMNIST(
    root=data_root,
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)
test_ds = datasets.FashionMNIST(
    root=data_root,
    train=False,
    download=True,
    transform=transforms.ToTensor(),
)

def ds_to_numpy(ds, max_samples):
    x = ds.data.numpy().astype(np.float64) / 255.0
    y = ds.targets.numpy().astype(np.int64)
    x = x[:max_samples].reshape(max_samples, -1)
    y = y[:max_samples]
    return x, y

X_train, y_train = ds_to_numpy(train_ds, 12000)
X_test, y_test = ds_to_numpy(test_ds, 2000)

n_classes = 10
y_train_oh = np.eye(n_classes)[y_train]

print("Train shape:", X_train.shape, y_train_oh.shape)
print("Test shape:", X_test.shape, y_test.shape)


In [ ]:
builder = ModelBuilder()
model = builder.Sequential(
    [
        builder.Dense(512, activation="relu", input_shape=(784,)),
        builder.Dropout(0.2),
        builder.Dense(256, activation="relu"),
        builder.Dense(10, activation="softmax"),
    ],
    name="fmnist_dense_baseline",
)

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    learning_rate=0.001,
    log_level="detailed",
)

history = model.fit(
    X_train,
    y_train_oh,
    epochs=8,
    batch_size=128,
    validation_split=0.1,
)


In [ ]:
preds = model.predict(X_test)
y_pred = np.argmax(preds, axis=1)
acc = float(np.mean(y_pred == y_test))

print(f"Test accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred, digits=3))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history["loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["accuracy"], label="train")
plt.plot(history["val_accuracy"], label="val")
plt.title("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()
